In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import os

In [2]:
def focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        y_true    = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_true_oh = tf.one_hot(y_true, depth=5)
        probs     = tf.reduce_sum(y_pred * y_true_oh, axis=-1)
        fw        = alpha * tf.pow(1.0 - probs, gamma)
        loss      = -fw * tf.math.log(probs + 1e-8)
        return tf.reduce_mean(loss)
    return loss_fn

In [3]:
def build_encoder(input_shape=(187, 1), name='encoder'):
    inputs = keras.Input(shape=input_shape, name=f'{name}_input')
    x = keras.layers.Conv1D(24, 5, padding='same', activation='relu')(inputs)
    x = keras.layers.Conv1D(24, 5, padding='same', activation='relu')(x)
    x = keras.layers.MaxPooling1D(2)(x)
    x = keras.layers.Conv1D(48, 3, padding='same', activation='relu')(x)
    x = keras.layers.Conv1D(48, 3, padding='same', activation='relu')(x)
    x = keras.layers.MaxPooling1D(2)(x)
    x = keras.layers.Conv1D(48, 3, padding='same', activation='relu')(x)
    return keras.Model(inputs, x, name=name)

class ChannelGate(keras.layers.Layer):
    def __init__(self, dim, **kw):
        super().__init__(**kw)
        self.d1 = keras.layers.Dense(dim, activation='relu')
        self.d2 = keras.layers.Dense(dim, activation='sigmoid')
    def call(self, query_feat, context_feat):
        ctx = tf.reduce_mean(context_feat, axis=1, keepdims=True)
        ctx = self.d1(ctx)
        gate = self.d2(ctx)
        return query_feat * gate

def build_fusion_model(input_shape=(187, 1), num_classes=5):
    ecg_input = keras.Input(shape=input_shape, name='ecg_input')
    ppg_input = keras.Input(shape=input_shape, name='ppg_input')
    ecg_feat = build_encoder(name='ecg_encoder')(ecg_input)
    ppg_feat = build_encoder(name='ppg_encoder')(ppg_input)
    ecg_x = ChannelGate(48, name='ecg_attends_ppg')(ecg_feat, ppg_feat)
    ppg_x = ChannelGate(48, name='ppg_attends_ecg')(ppg_feat, ecg_feat)
    fused = keras.layers.Add()([ecg_x, ppg_x])
    fused = keras.layers.BatchNormalization()(fused)
    fused = keras.layers.GlobalAveragePooling1D()(fused)
    x = keras.layers.Dense(48, activation='relu')(fused)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model([ecg_input, ppg_input], outputs, name='ecg_ppg_fusion')

model = build_fusion_model()
model.load_weights('models/fusion_weights.weights.h5')
print("Model rebuilt and weights loaded.")


Model rebuilt and weights loaded.


In [4]:
size_kb = os.path.getsize('models/fusion_model.keras') / 1024
print(f"Original model size: {size_kb:.1f} KB")

Original model size: 752.8 KB


In [5]:
X_test_ecg = np.load('data/X_test.npy')
X_test_ppg = np.load('data/PPG_pairs.npy')

# NN matching for test PPG (same as training)
from sklearn.neighbors import NearestNeighbors
ECG_pairs = np.load('data/ECG_pairs.npy')
PPG_pairs = np.load('data/PPG_pairs.npy')

bidmc_flat = ECG_pairs.reshape(len(ECG_pairs), -1)
nn = NearestNeighbors(n_neighbors=1, metric='euclidean', n_jobs=-1)
nn.fit(bidmc_flat)

test_flat = X_test_ecg.reshape(len(X_test_ecg), -1)
_, test_idx = nn.kneighbors(test_flat)
X_test_ppg = PPG_pairs[test_idx.squeeze()]

print(f"ECG test: {X_test_ecg.shape}")
print(f"PPG test: {X_test_ppg.shape}")

# Use 200 samples as calibration dataset for quantization
calib_ecg = X_test_ecg[:200].astype(np.float32)
calib_ppg = X_test_ppg[:200].astype(np.float32)
print("Calibration data ready")

ECG test: (21892, 187, 1)
PPG test: (21892, 187, 1)
Calibration data ready


In [6]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_dynamic = converter.convert()
with open('models/fusion_model_dynamic.tflite', 'wb') as f:
    f.write(tflite_dynamic)

interp = tf.lite.Interpreter(model_content=tflite_dynamic)
interp.allocate_tensors()
ops = sorted(set(o['op_name'] for o in interp._get_ops_details()))
print("Size KB:", round(len(tflite_dynamic)/1024,1))
print("OPS:", ops)

INFO:tensorflow:Assets written to: C:\Users\tagui\AppData\Local\Temp\tmp6x4pqvk3\assets


INFO:tensorflow:Assets written to: C:\Users\tagui\AppData\Local\Temp\tmp6x4pqvk3\assets


Saved artifact at 'C:\Users\tagui\AppData\Local\Temp\tmp6x4pqvk3'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 187, 1), dtype=tf.float32, name='ecg_input'), TensorSpec(shape=(None, 187, 1), dtype=tf.float32, name='ppg_input')]
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  2042004544528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2042004546064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2042004547024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2042004546256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2042004547408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2042004547216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2042004546448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2042004546832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2042004546640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  20420

c:\Users\tagui\Downloads\ECG_PPG\pfe_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [7]:
""" from sklearn.metrics import classification_report, f1_score

interp_dyn = tf.lite.Interpreter(model_content=tflite_dynamic)
interp_dyn.allocate_tensors()
in_d  = interp_dyn.get_input_details()
out_d = interp_dyn.get_output_details()

ecg_idx = next(i for i, d in enumerate(in_d) if 'ecg' in d['name'])
ppg_idx = next(i for i, d in enumerate(in_d) if 'ppg' in d['name'])

y_pred_dyn = []
for i in range(len(X_test_ecg)):
    interp_dyn.set_tensor(in_d[ecg_idx]['index'], X_test_ecg[i:i+1].astype(np.float32))
    interp_dyn.set_tensor(in_d[ppg_idx]['index'], X_test_ppg[i:i+1].astype(np.float32))
    interp_dyn.invoke()
    y_pred_dyn.append(np.argmax(interp_dyn.get_tensor(out_d[0]['index'])))
y_pred_dyn = np.array(y_pred_dyn)

class_names = ['N Normal', 'S Supra', 'V Ventricular', 'F Fusion', 'Q Unknown']
print("=== DYNAMIC-RANGE MODEL RESULTS ===")
print(classification_report(y_test, y_pred_dyn, target_names=class_names, digits=4))
print("Macro F1:", round(f1_score(y_test, y_pred_dyn, average='macro'), 4)) """

' from sklearn.metrics import classification_report, f1_score\n\ninterp_dyn = tf.lite.Interpreter(model_content=tflite_dynamic)\ninterp_dyn.allocate_tensors()\nin_d  = interp_dyn.get_input_details()\nout_d = interp_dyn.get_output_details()\n\necg_idx = next(i for i, d in enumerate(in_d) if \'ecg\' in d[\'name\'])\nppg_idx = next(i for i, d in enumerate(in_d) if \'ppg\' in d[\'name\'])\n\ny_pred_dyn = []\nfor i in range(len(X_test_ecg)):\n    interp_dyn.set_tensor(in_d[ecg_idx][\'index\'], X_test_ecg[i:i+1].astype(np.float32))\n    interp_dyn.set_tensor(in_d[ppg_idx][\'index\'], X_test_ppg[i:i+1].astype(np.float32))\n    interp_dyn.invoke()\n    y_pred_dyn.append(np.argmax(interp_dyn.get_tensor(out_d[0][\'index\'])))\ny_pred_dyn = np.array(y_pred_dyn)\n\nclass_names = [\'N Normal\', \'S Supra\', \'V Ventricular\', \'F Fusion\', \'Q Unknown\']\nprint("=== DYNAMIC-RANGE MODEL RESULTS ===")\nprint(classification_report(y_test, y_pred_dyn, target_names=class_names, digits=4))\nprint("Macro 

In [8]:
# Float32 export for ESP32 deployment (plain CONV_2D — invokable on TFLite Micro)
conv = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_fp32 = conv.convert()
with open('models/fusion_model_fp32.tflite', 'wb') as f:
    f.write(tflite_fp32)

# Generate .h from the float32 model
with open('models/fusion_model_fp32.tflite', 'rb') as f:
    data = f.read()
arr = ', '.join(f'0x{b:02x}' for b in data)
code = f'''#ifndef FUSION_MODEL_H
#define FUSION_MODEL_H
const unsigned char fusion_model[] = {{
  {arr}
}};
const unsigned int fusion_model_len = {len(data)};
#endif
'''
open('models/fusion_model.h', 'w').write(code)
print("float32 .h saved —", round(len(data)/1024, 1), "KB")

INFO:tensorflow:Assets written to: C:\Users\tagui\AppData\Local\Temp\tmp4cmav24w\assets


INFO:tensorflow:Assets written to: C:\Users\tagui\AppData\Local\Temp\tmp4cmav24w\assets


Saved artifact at 'C:\Users\tagui\AppData\Local\Temp\tmp4cmav24w'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 187, 1), dtype=tf.float32, name='ecg_input'), TensorSpec(shape=(None, 187, 1), dtype=tf.float32, name='ppg_input')]
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  2042004544528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2042004546064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2042004547024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2042004546256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2042004547408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2042004547216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2042004546448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2042004546832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2042004546640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  20420

In [11]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import classification_report, f1_score

# Load the same data notebook 07 used
X_test_ecg = np.load('data/X_test.npy')
y_test     = np.load('data/y_test.npy')
ECG_pairs  = np.load('data/ECG_pairs.npy')
PPG_pairs  = np.load('data/PPG_pairs.npy')

# Rebuild X_test_ppg via the SAME nearest-neighbor matching as nb07
nn = NearestNeighbors(n_neighbors=1, metric='euclidean', n_jobs=-1)
nn.fit(ECG_pairs.reshape(len(ECG_pairs), -1))
_, test_idx = nn.kneighbors(X_test_ecg.reshape(len(X_test_ecg), -1))
X_test_ppg = PPG_pairs[test_idx.squeeze()]

# Evaluate the DEPLOYED float32 tflite model
interp = tf.lite.Interpreter(model_path='models/fusion_model_fp32.tflite')
interp.allocate_tensors()
in_d  = interp.get_input_details()
out_d = interp.get_output_details()
ecg_idx = next(i for i, d in enumerate(in_d) if 'ecg' in d['name'])
ppg_idx = next(i for i, d in enumerate(in_d) if 'ppg' in d['name'])

y_pred = []
for i in range(len(X_test_ecg)):
    interp.set_tensor(in_d[ecg_idx]['index'], X_test_ecg[i:i+1].astype(np.float32))
    interp.set_tensor(in_d[ppg_idx]['index'], X_test_ppg[i:i+1].astype(np.float32))
    interp.invoke()
    y_pred.append(np.argmax(interp.get_tensor(out_d[0]['index'])))
y_pred = np.array(y_pred)

class_names = ['N Normal', 'S Supra', 'V Ventricular', 'F Fusion', 'Q Unknown']
print("=== DEPLOYED FLOAT32 MODEL (ESP32) RESULTS ===")
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))
print("Macro F1:", round(f1_score(y_test, y_pred, average='macro'), 4))

c:\Users\tagui\Downloads\ECG_PPG\pfe_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


=== DEPLOYED FLOAT32 MODEL (ESP32) RESULTS ===
               precision    recall  f1-score   support

     N Normal     0.9792    0.9949    0.9870     18118
      S Supra     0.9049    0.5306    0.6689       556
V Ventricular     0.9346    0.9482    0.9414      1448
     F Fusion     0.8772    0.6173    0.7246       162
    Q Unknown     0.9917    0.9714    0.9815      1608

     accuracy                         0.9755     21892
    macro avg     0.9375    0.8125    0.8607     21892
 weighted avg     0.9746    0.9755    0.9736     21892

Macro F1: 0.8607


In [12]:
import numpy as np
import tensorflow as tf

# Charger le modèle INT8 déjà sauvegardé
interpreter = tf.lite.Interpreter(model_path='models/fusion_model_int8.tflite')
interpreter.allocate_tensors()

# Charger les données de test
X_test = np.load('data/X_test.npy')
y_test = np.load('data/y_test.npy')

# Vérifier les inputs attendus
input_details = interpreter.get_input_details()
for i, inp in enumerate(input_details):
    print(f"Input {i}: name={inp['name']}, shape={inp['shape']}, dtype={inp['dtype']}")

# Inférence sur un sous-ensemble (sinon trop long)
from sklearn.metrics import f1_score
import numpy as np

n_samples = len(X_test)
preds = []

for i in range(n_samples):
    ecg = X_test[i:i+1].astype(np.float32)
    ppg = X_test[i:i+1].astype(np.float32)  # proxy PPG
    
    for detail in input_details:
        if 'ecg' in detail['name'].lower():
            interpreter.set_tensor(detail['index'], ecg)
        else:
            interpreter.set_tensor(detail['index'], ppg)
    
    interpreter.invoke()
    output = interpreter.get_output_details()[0]
    pred = interpreter.get_tensor(output['index'])
    preds.append(np.argmax(pred))

preds = np.array(preds)
f1 = f1_score(y_test, preds, average='macro')
print(f"\nINT8 Macro F1: {f1:.4f}")
print(f"INT8 Accuracy: {(preds == y_test).mean():.4f}")

c:\Users\tagui\Downloads\ECG_PPG\pfe_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Input 0: name=serving_default_ppg_input:0, shape=[  1 187   1], dtype=<class 'numpy.float32'>
Input 1: name=serving_default_ecg_input:0, shape=[  1 187   1], dtype=<class 'numpy.float32'>

INT8 Macro F1: 0.8129
INT8 Accuracy: 0.9366


In [13]:
import os
print(round(os.path.getsize('models/fusion_model_fp32.tflite')/1024, 1), "KB")

229.9 KB
